# Porto Seguro Experiment Notebook

This notebook generates and analyses the Porto Seguro's Safe Driver Prediction experiment used in the cross-validation reliability research project.

The purpose of this notebook is not to maximise Kaggle leaderboard rank. Instead, it creates a controlled empirical model-evaluation dataset by training many model configurations and recording their validation behaviour.

## Main Objectives

This notebook performs the following steps:

1. Load the Porto Seguro training, test, and sample submission files.
2. Replace Porto Seguro missing-value markers (`-1`) with `NaN`.
3. Generate 1,000 controlled model configurations across multiple model families.
4. Evaluate each configuration using stratified 5-fold cross-validation.
5. Create Kaggle-compatible submission files.
6. Construct a representative 100-model leaderboard subset.
7. Convert Kaggle Normalized Gini leaderboard scores to AUC.
8. Merge manually verified public and private leaderboard scores.
9. Export the final Porto research dataset used in the Bayesian melding analysis.

## Final Outputs

The main outputs are saved under:

`results/porto/`

Key output files include:

- `porto_1000_model_configs.csv`
- `porto_1000_model_results.csv`
- `manual_submission_plan_100.csv`
- `porto_leaderboard_scores.csv`
- `porto_final_research_dataset.csv`
- `porto_bayesian_melding_input.csv`

## Reproducibility Note

Raw Kaggle competition data is not included in this repository. To reproduce the notebook, download the Porto Seguro's Safe Driver Prediction dataset from Kaggle and place the files in:

`data/porto/`

## 1. Setup

This section imports the required Python libraries and sets the random seed used for reproducible model configuration generation.

In [1]:
import os
import json
import time
import warnings
import subprocess
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import roc_auc_score

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. Project Paths

This section defines the local project paths used by the notebook.

Raw Kaggle data is expected under `data/porto/`.

Experiment outputs are written to `results/porto/`.

Generated submission files are written to `submissions/porto/`, which is ignored by Git because submission CSVs can be large and are not required for the final research analysis.

In [2]:
PROJECT_ROOT = Path("..")

DATA_DIR = PROJECT_ROOT / "data" / "porto"
SUBMISSION_DIR = PROJECT_ROOT / "submissions" / "porto"
RESULTS_DIR = PROJECT_ROOT / "results" / "porto"

SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_SUB_PATH = DATA_DIR / "sample_submission.csv"

COMPETITION_NAME = "porto-seguro-safe-driver-prediction"

RESULTS_CSV = RESULTS_DIR / "porto_1000_model_results.csv"
CONFIG_CSV = RESULTS_DIR / "porto_1000_model_configs.csv"

print("Project root:", PROJECT_ROOT)
print("Data dir:", DATA_DIR)
print("Submission dir:", SUBMISSION_DIR)
print("Results dir:", RESULTS_DIR)

Project root: ..
Data dir: ../data/porto
Submission dir: ../submissions/porto
Results dir: ../results/porto


## 3. Experiment Controls

This section controls which part of the 1,000-model configuration list is executed.

For repository safety, the default range is empty:

```python
START_INDEX = 0
END_INDEX = 0
```

To reproduce a small test batch, manually change the values to something like:

```python
START_INDEX = 0
END_INDEX = 5
```

Kaggle auto-submission is disabled by default. Do not submit large batches automatically, as Kaggle may enforce submission limits or rate throttling.

In [16]:
# Default to an empty run for repository safety.
# Set START_INDEX and END_INDEX manually when reproducing experiments.
START_INDEX = 0
END_INDEX = 0

AUTO_SUBMIT = False
MAX_SUBMISSIONS_PER_RUN = 1

# 5-fold CV for all generated models.
cv_5 = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

print("START_INDEX:", START_INDEX)
print("END_INDEX:", END_INDEX)
print("AUTO_SUBMIT:", AUTO_SUBMIT)
print("MAX_SUBMISSIONS_PER_RUN:", MAX_SUBMISSIONS_PER_RUN)

START_INDEX: 5
END_INDEX: 1000
AUTO_SUBMIT: False
MAX_SUBMISSIONS_PER_RUN: 1


## 4. Load Porto Seguro Dataset

In [4]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Sample submission shape:", sample_sub.shape)

display(train_df.head())
display(sample_sub.head())

Train shape: (595212, 59)
Test shape: (892816, 58)
Sample submission shape: (892816, 2)


,id,target,ps_ind_01,ps_ind_02_cat,ps_ind_03,ps_ind_04_cat,ps_ind_05_cat,ps_ind_06_bin,ps_ind_07_bin,ps_ind_08_bin,...,ps_calc_11,ps_calc_12,ps_calc_13,ps_calc_14,ps_calc_15_bin,ps_calc_16_bin,ps_calc_17_bin,ps_calc_18_bin,ps_calc_19_bin,ps_calc_20_bin
0,7,0,2,2,5,1,0,0,1,0,...,9,1,5,8,0,1,1,0,0,1
1,9,0,1,1,7,0,0,0,0,1,...,3,1,1,9,0,1,1,0,1,0
2,13,0,5,4,9,1,0,0,0,1,...,4,2,7,7,0,1,1,0,1,0
3,16,0,0,1,2,0,0,1,0,0,...,2,2,4,9,0,0,0,0,0,0
4,17,0,0,2,0,1,0,1,0,0,...,3,1,1,3,0,0,0,1,1,0


,id,target
0,0,0.0364
1,1,0.0364
2,2,0.0364
3,3,0.0364
4,4,0.0364


In [5]:
TARGET_COL = "target"
ID_COL = "id"

feature_cols = [c for c in train_df.columns if c not in [ID_COL, TARGET_COL]]

X = train_df[feature_cols].copy()
y = train_df[TARGET_COL].copy()
X_test = test_df[feature_cols].copy()

# Porto Seguro uses -1 to represent missing values in many columns.
# Convert -1 to NaN so the imputer handles them properly.
X = X.replace(-1, np.nan)
X_test = X_test.replace(-1, np.nan)

print("Number of features:", len(feature_cols))
print("X shape:", X.shape)
print("y shape:", y.shape)
print("X_test shape:", X_test.shape)

print("\nTarget distribution:")
print(y.value_counts(normalize=True))

print("\nSample submission columns:")
print(sample_sub.columns.tolist())

Number of features: 57
X shape: (595212, 57)
y shape: (595212,)
X_test shape: (892816, 57)

Target distribution:
target
0    0.963552
1    0.036448
Name: proportion, dtype: float64

Sample submission columns:
['id', 'target']


## 5. Generate 1000 Controlled Model Configurations

The experiment uses five model families:

- LightGBM: 350 configurations
- XGBoost: 350 configurations
- CatBoost: 150 configurations
- Random Forest / ExtraTrees: 75 configurations
- Logistic Regression: 75 configurations

This gives 1000 model configurations for the Porto dataset.

In [6]:
def sample_lgbm_config(rng, model_id):
    return {
        "model_id": model_id,
        "model_family": "lightgbm",
        "seed": int(rng.integers(1, 1_000_000)),
        "params": {
            "n_estimators": int(rng.choice([300, 500, 700, 900, 1100])),
            "learning_rate": float(rng.choice([0.01, 0.02, 0.03, 0.05, 0.07])),
            "num_leaves": int(rng.choice([15, 31, 63, 127])),
            "max_depth": int(rng.choice([-1, 3, 5, 7, 9])),
            "min_child_samples": int(rng.choice([10, 20, 40, 80, 120])),
            "subsample": float(rng.choice([0.6, 0.7, 0.8, 0.9, 1.0])),
            "colsample_bytree": float(rng.choice([0.5, 0.6, 0.7, 0.8, 0.9, 1.0])),
            "reg_alpha": float(rng.choice([0.0, 0.01, 0.1, 0.5, 1.0])),
            "reg_lambda": float(rng.choice([0.0, 0.1, 0.5, 1.0, 2.0, 5.0])),
        }
    }


def sample_xgb_config(rng, model_id):
    return {
        "model_id": model_id,
        "model_family": "xgboost",
        "seed": int(rng.integers(1, 1_000_000)),
        "params": {
            "n_estimators": int(rng.choice([300, 500, 700, 900, 1100])),
            "learning_rate": float(rng.choice([0.01, 0.02, 0.03, 0.05, 0.07])),
            "max_depth": int(rng.choice([3, 4, 5, 6, 7])),
            "min_child_weight": float(rng.choice([1, 2, 3, 5, 8, 10])),
            "gamma": float(rng.choice([0.0, 0.01, 0.05, 0.1, 0.3])),
            "subsample": float(rng.choice([0.6, 0.7, 0.8, 0.9, 1.0])),
            "colsample_bytree": float(rng.choice([0.5, 0.6, 0.7, 0.8, 0.9, 1.0])),
            "reg_alpha": float(rng.choice([0.0, 0.01, 0.1, 0.5, 1.0])),
            "reg_lambda": float(rng.choice([0.1, 0.5, 1.0, 2.0, 5.0])),
        }
    }


def sample_catboost_config(rng, model_id):
    return {
        "model_id": model_id,
        "model_family": "catboost",
        "seed": int(rng.integers(1, 1_000_000)),
        "params": {
            "iterations": int(rng.choice([300, 500, 700, 900])),
            "learning_rate": float(rng.choice([0.01, 0.02, 0.03, 0.05, 0.07])),
            "depth": int(rng.choice([4, 5, 6, 7, 8])),
            "l2_leaf_reg": float(rng.choice([1, 3, 5, 7, 10])),
            "random_strength": float(rng.choice([0.5, 1.0, 2.0, 5.0])),
            "bagging_temperature": float(rng.choice([0.0, 0.5, 1.0, 2.0])),
        }
    }


def sample_tree_config(rng, model_id):
    model_family = str(rng.choice(["random_forest", "extra_trees"]))

    return {
        "model_id": model_id,
        "model_family": model_family,
        "seed": int(rng.integers(1, 1_000_000)),
        "params": {
            "n_estimators": int(rng.choice([200, 300, 500])),
            "max_depth": rng.choice([None, 5, 8, 12, 16, 24]).item() if hasattr(rng.choice([None, 5]), "item") else None,
            "min_samples_split": int(rng.choice([2, 5, 10, 20])),
            "min_samples_leaf": int(rng.choice([1, 2, 4, 8])),
            "max_features": str(rng.choice(["sqrt", "log2"])),
            "bootstrap": bool(rng.choice([True, False])) if model_family == "random_forest" else False,
        }
    }


def sample_logreg_config(rng, model_id):
    penalty = str(rng.choice(["l2", "l1"]))
    solver = "liblinear" if penalty == "l1" else str(rng.choice(["lbfgs", "liblinear"]))

    return {
        "model_id": model_id,
        "model_family": "logistic_regression",
        "seed": int(rng.integers(1, 1_000_000)),
        "params": {
            "C": float(rng.choice([0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0])),
            "penalty": penalty,
            "solver": solver,
            "max_iter": 1000,
        }
    }

In [7]:
def create_model_configs(random_state=42):
    rng = np.random.default_rng(random_state)
    configs = []
    model_id = 0

    # LightGBM: 350
    for _ in range(350):
        configs.append(sample_lgbm_config(rng, model_id))
        model_id += 1

    # XGBoost: 350
    for _ in range(350):
        configs.append(sample_xgb_config(rng, model_id))
        model_id += 1

    # CatBoost: 150
    for _ in range(150):
        configs.append(sample_catboost_config(rng, model_id))
        model_id += 1

    # Random Forest / ExtraTrees: 75
    for _ in range(75):
        configs.append(sample_tree_config(rng, model_id))
        model_id += 1

    # Logistic Regression: 75
    for _ in range(75):
        configs.append(sample_logreg_config(rng, model_id))
        model_id += 1

    return configs


model_configs = create_model_configs(RANDOM_STATE)

print("Total configs:", len(model_configs))
pd.DataFrame([
    {
        "model_id": c["model_id"],
        "model_family": c["model_family"],
        "seed": c["seed"],
        "params_json": json.dumps(c["params"])
    }
    for c in model_configs
]).to_csv(CONFIG_CSV, index=False)

print("Saved config table to:", CONFIG_CSV)

display(pd.DataFrame(model_configs).head())

Total configs: 1000
Saved config table to: ../results/porto/porto_1000_model_configs.csv


,model_id,model_family,seed,params
0,0,lightgbm,89251,"{'n_estimators': 900, 'learning_rate': 0.05, '..."
1,1,lightgbm,526479,"{'n_estimators': 1100, 'learning_rate': 0.05, ..."
2,2,lightgbm,500352,"{'n_estimators': 500, 'learning_rate': 0.01, '..."
3,3,lightgbm,450460,"{'n_estimators': 500, 'learning_rate': 0.01, '..."
4,4,lightgbm,165229,"{'n_estimators': 900, 'learning_rate': 0.05, '..."


In [8]:
# Clean max_depth values in tree-based sklearn configs.
# This prevents weird numpy object issues from random sampling.

for config in model_configs:
    if config["model_family"] in ["random_forest", "extra_trees"]:
        md = config["params"].get("max_depth", None)
        if pd.isna(md):
            config["params"]["max_depth"] = None
        elif md is not None:
            config["params"]["max_depth"] = int(md)

## 6. Build Model Pipelines

Each model family uses a consistent preprocessing pipeline.

- Tree/boosting models use median imputation.
- Logistic Regression uses median imputation + standard scaling.

In [9]:
def build_pipeline(config):
    family = config["model_family"]
    params = config["params"].copy()
    seed = config["seed"]

    if family == "lightgbm":
        model = LGBMClassifier(
            **params,
            objective="binary",
            random_state=seed,
            n_jobs=-1,
            verbosity=-1
        )

        pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", model)
        ])

    elif family == "xgboost":
        model = XGBClassifier(
            **params,
            objective="binary:logistic",
            eval_metric="auc",
            random_state=seed,
            n_jobs=-1,
            tree_method="hist"
        )

        pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", model)
        ])

    elif family == "catboost":
        model = CatBoostClassifier(
            **params,
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=seed,
            verbose=False,
            allow_writing_files=False
        )

        pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", model)
        ])

    elif family == "random_forest":
        model = RandomForestClassifier(
            **params,
            random_state=seed,
            n_jobs=-1,
            class_weight=None
        )

        pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", model)
        ])

    elif family == "extra_trees":
        params.pop("bootstrap", None)

        model = ExtraTreesClassifier(
            **params,
            random_state=seed,
            n_jobs=-1,
            class_weight=None
        )

        pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", model)
        ])

    elif family == "logistic_regression":
        model = LogisticRegression(
            **params,
            random_state=seed
        )

        pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", model)
        ])

    else:
        raise ValueError(f"Unknown model family: {family}")

    return pipeline

## 7. Kaggle Submission Helpers

These functions:

1. Create a submission CSV
2. Submit the CSV to Kaggle if `AUTO_SUBMIT=True`
3. Read the latest Kaggle public/private score
4. Return those scores for storage in the results CSV

Important: submit in small batches only.

In [21]:
def run_shell_command(command, timeout=300):
    """
    Run a shell command and return stdout/stderr.
    """
    try:
        completed = subprocess.run(
            command,
            shell=True,
            capture_output=True,
            text=True,
            timeout=timeout
        )
        return completed.returncode, completed.stdout, completed.stderr
    except subprocess.TimeoutExpired:
        return 1, "", "Command timed out"


def create_submission(model_id, pipeline, X, y, X_test, sample_sub, submission_dir):
    """
    Fit model on full training data and save test predictions.
    """
    pipeline.fit(X, y)

    test_pred = pipeline.predict_proba(X_test)[:, 1]

    submission = sample_sub.copy()
    submission["target"] = test_pred

    submission_path = submission_dir / f"porto_model_{model_id:04d}.csv"
    submission.to_csv(submission_path, index=False)

    return submission_path


def submit_to_kaggle(submission_path, model_id, model_family):
    """
    Submit a CSV to Kaggle.
    """
    message = f"Porto model {model_id:04d} {model_family}"

    command = (
        f'kaggle competitions submit '
        f'-c {COMPETITION_NAME} '
        f'-f "{submission_path}" '
        f'-m "{message}"'
    )

    returncode, stdout, stderr = run_shell_command(command, timeout=600)

    success = returncode == 0 and "Successfully submitted" in stdout

    return {
        "submitted_to_kaggle": success,
        "submit_stdout": stdout,
        "submit_stderr": stderr
    }


def fetch_kaggle_submissions():
    """
    Fetch Kaggle submissions table using the Kaggle CLI.
    """
    command = f"kaggle competitions submissions -c {COMPETITION_NAME}"
    returncode, stdout, stderr = run_shell_command(command, timeout=300)

    if returncode != 0:
        print("Could not fetch submissions.")
        print(stderr)
        return None

    print(stdout)
    return stdout


def parse_latest_scores_from_cli_output(stdout, filename):
    """
    Parse public/private scores from Kaggle CLI output.

    This is a simple parser based on matching the file name line.
    It may need adjustment if Kaggle CLI output format changes.
    """
    if stdout is None:
        return None, None, "NO_OUTPUT"

    lines = stdout.splitlines()

    matching_lines = [line for line in lines if filename in line]

    if len(matching_lines) == 0:
        return None, None, "NOT_FOUND"

    line = matching_lines[0]
    parts = line.split()

    # Public and private scores are usually the last two columns.
    try:
        public_gini = float(parts[-2])
        private_gini = float(parts[-1])

        public_score = (public_gini + 1) / 2
        private_score = (private_gini + 1) / 2
        status = "COMPLETE"
    except Exception:
        public_score = None
        private_score = None
        status = "PARSE_FAILED"

    return public_score, private_score, status

## 8. Evaluation Function

For each model configuration, we calculate 5-fold CV AUC.

Then we fit on the full training set and create a submission file.

If `AUTO_SUBMIT=True`, the notebook submits the file to Kaggle and tries to collect public/private scores automatically.

In [11]:
def evaluate_single_config(config, X, y, X_test, sample_sub, auto_submit=False):
    model_id = config["model_id"]
    family = config["model_family"]
    seed = config["seed"]
    params = config["params"]

    print(f"\nRunning model_id={model_id}, family={family}")

    pipeline = build_pipeline(config)

    start_time = time.time()

    result = {
        "competition": "porto",
        "model_id": model_id,
        "model_family": family,
        "seed": seed,
        "params_json": json.dumps(params),
        "started_at": datetime.now().isoformat(),
        "cv_auc_mean": None,
        "cv_auc_std": None,
        "cv_auc_min": None,
        "cv_auc_max": None,
        "submission_path": None,
        "submitted_to_kaggle": False,
        "kaggle_status": None,
        "public_auc": None,
        "private_auc": None,
        "runtime_seconds": None,
        "error": None
    }

    try:
        cv_scores = cross_val_score(
            pipeline,
            X,
            y,
            scoring="roc_auc",
            cv=cv_5,
            n_jobs=-1
        )

        result["cv_auc_mean"] = float(np.mean(cv_scores))
        result["cv_auc_std"] = float(np.std(cv_scores))
        result["cv_auc_min"] = float(np.min(cv_scores))
        result["cv_auc_max"] = float(np.max(cv_scores))

        print("CV AUC mean:", result["cv_auc_mean"])
        print("CV AUC std:", result["cv_auc_std"])

        submission_path = create_submission(
            model_id=model_id,
            pipeline=pipeline,
            X=X,
            y=y,
            X_test=X_test,
            sample_sub=sample_sub,
            submission_dir=SUBMISSION_DIR
        )

        result["submission_path"] = str(submission_path)

        if auto_submit:
            submit_result = submit_to_kaggle(
                submission_path=submission_path,
                model_id=model_id,
                model_family=family
            )

            result["submitted_to_kaggle"] = bool(submit_result["submitted_to_kaggle"])

            if result["submitted_to_kaggle"]:
                print("Submitted successfully. Waiting before fetching score...")
                time.sleep(90)

                stdout = fetch_kaggle_submissions()

                public_auc, private_auc, status = parse_latest_scores_from_cli_output(
                    stdout=stdout,
                    filename=submission_path.name
                )

                result["public_auc"] = public_auc
                result["private_auc"] = private_auc
                result["kaggle_status"] = status
            else:
                result["kaggle_status"] = "SUBMISSION_FAILED"
                result["error"] = submit_result["submit_stderr"]

    except Exception as e:
        result["error"] = repr(e)
        print("ERROR:", repr(e))

    result["runtime_seconds"] = float(time.time() - start_time)
    result["finished_at"] = datetime.now().isoformat()

    return result

## 9. Resume Existing Results

This cell loads existing results if the notebook has already been run before.

This allows the experiment to resume without repeating completed models.

In [12]:
if RESULTS_CSV.exists():
    existing_results_df = pd.read_csv(RESULTS_CSV)
    completed_ids = set(existing_results_df["model_id"].astype(int).tolist())
    print("Loaded existing results:", existing_results_df.shape)
    print("Completed model ids:", len(completed_ids))
else:
    existing_results_df = pd.DataFrame()
    completed_ids = set()
    print("No existing results found.")

No existing results found.


## 10. Run Experiment Batch

Start with a small batch.

Recommended first run:

```python
START_INDEX = 0
END_INDEX = 5
AUTO_SUBMIT = False
```

After confirming everything works, increase the range.

When submitting to Kaggle, keep `MAX_SUBMISSIONS_PER_RUN` small.

In [17]:
batch_configs = model_configs[START_INDEX:END_INDEX]

new_results = []
submission_count = 0

for config in batch_configs:
    model_id = int(config["model_id"])

    if model_id in completed_ids:
        print(f"Skipping model_id={model_id}; already completed.")
        continue

    should_submit = AUTO_SUBMIT and (submission_count < MAX_SUBMISSIONS_PER_RUN)

    result = evaluate_single_config(
        config=config,
        X=X,
        y=y,
        X_test=X_test,
        sample_sub=sample_sub,
        auto_submit=should_submit
    )

    if should_submit:
        submission_count += 1

    new_results.append(result)

    # Save after every model so progress is not lost.
    temp_df = pd.DataFrame(new_results)

    if RESULTS_CSV.exists():
        old_df = pd.read_csv(RESULTS_CSV)
        combined_df = pd.concat([old_df, temp_df], ignore_index=True)
        combined_df = combined_df.drop_duplicates(subset=["model_id"], keep="last")
    else:
        combined_df = temp_df.copy()

    combined_df.to_csv(RESULTS_CSV, index=False)

    print(f"Saved progress to {RESULTS_CSV}")

print("Batch complete.")
print("New results:", len(new_results))


Running model_id=5, family=lightgbm
CV AUC mean: 0.639732390537518
CV AUC std: 0.003432726541785231
Saved progress to ../results/porto/porto_1000_model_results.csv

Running model_id=6, family=lightgbm
CV AUC mean: 0.6396008687008461
CV AUC std: 0.002826399031353025
Saved progress to ../results/porto/porto_1000_model_results.csv

Running model_id=7, family=lightgbm
CV AUC mean: 0.6397548594761
CV AUC std: 0.003333854683384517
Saved progress to ../results/porto/porto_1000_model_results.csv

Running model_id=8, family=lightgbm
CV AUC mean: 0.6407801028235454
CV AUC std: 0.0030656300906856497
Saved progress to ../results/porto/porto_1000_model_results.csv

Running model_id=9, family=lightgbm
CV AUC mean: 0.6255242816600168
CV AUC std: 0.0024330821989212285
Saved progress to ../results/porto/porto_1000_model_results.csv

Running model_id=10, family=lightgbm
CV AUC mean: 0.6395723620148408
CV AUC std: 0.003301717433168195
Saved progress to ../results/porto/porto_1000_model_results.csv

Runn

python(80942) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


CV AUC mean: 0.6384700081745012
CV AUC std: 0.0023287377711762097
Saved progress to ../results/porto/porto_1000_model_results.csv

Running model_id=48, family=lightgbm
CV AUC mean: 0.6387618674335565
CV AUC std: 0.0022777259718409324
Saved progress to ../results/porto/porto_1000_model_results.csv

Running model_id=49, family=lightgbm
CV AUC mean: 0.639080207945848
CV AUC std: 0.0035803027992730636
Saved progress to ../results/porto/porto_1000_model_results.csv

Running model_id=50, family=lightgbm
CV AUC mean: 0.6380455303139556
CV AUC std: 0.003025855030014019
Saved progress to ../results/porto/porto_1000_model_results.csv

Running model_id=51, family=lightgbm
CV AUC mean: 0.6092294402168312
CV AUC std: 0.0023461589463354784
Saved progress to ../results/porto/porto_1000_model_results.csv

Running model_id=52, family=lightgbm
CV AUC mean: 0.6281829257120033
CV AUC std: 0.003371901758338294
Saved progress to ../results/porto/porto_1000_model_results.csv

Running model_id=53, family=ligh

python(82132) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


CV AUC mean: 0.6399226531894536
CV AUC std: 0.0035653670944763975
Saved progress to ../results/porto/porto_1000_model_results.csv

Running model_id=71, family=lightgbm
CV AUC mean: 0.6215435890968524
CV AUC std: 0.0022408797612734495
Saved progress to ../results/porto/porto_1000_model_results.csv

Running model_id=72, family=lightgbm
CV AUC mean: 0.6356769420480426
CV AUC std: 0.0032124111932807357
Saved progress to ../results/porto/porto_1000_model_results.csv

Running model_id=73, family=lightgbm
CV AUC mean: 0.6396700443983804
CV AUC std: 0.003372493672118277
Saved progress to ../results/porto/porto_1000_model_results.csv

Running model_id=74, family=lightgbm
CV AUC mean: 0.6338918792051114
CV AUC std: 0.002685830218125462
Saved progress to ../results/porto/porto_1000_model_results.csv

Running model_id=75, family=lightgbm
CV AUC mean: 0.6357092999409495
CV AUC std: 0.003620347943047805
Saved progress to ../results/porto/porto_1000_model_results.csv

Running model_id=76, family=ligh

...

CV AUC mean: 0.6165617087423032
CV AUC std: 0.0018530184670400894
Saved progress to ../results/porto/porto_1000_model_results.csv

Running model_id=925, family=logistic_regression


python(9207) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(9211) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(9212) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


CV AUC mean: 0.6197291296082545
CV AUC std: 0.0039033851169320913
Saved progress to ../results/porto/porto_1000_model_results.csv

Running model_id=926, family=logistic_regression
CV AUC mean: 0.6200450365673584
CV AUC std: 0.004107202780596889
Saved progress to ../results/porto/porto_1000_model_results.csv

Running model_id=927, family=logistic_regression
CV AUC mean: 0.6151289165135052
CV AUC std: 0.003549669518454273
Saved progress to ../results/porto/porto_1000_model_results.csv

Running model_id=928, family=logistic_regression
CV AUC mean: 0.6198939529296543
CV AUC std: 0.004041822199998521
Saved progress to ../results/porto/porto_1000_model_results.csv

Running model_id=929, family=logistic_regression
CV AUC mean: 0.620361250138948
CV AUC std: 0.004108442522579077
Saved progress to ../results/porto/porto_1000_model_results.csv

Running model_id=930, family=logistic_regression
CV AUC mean: 0.6198448021560529
CV AUC std: 0.004005861090229701
Saved progress to ../results/porto/porto

python(9822) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


CV AUC mean: 0.6203613722912861
CV AUC std: 0.00410842965606835
Saved progress to ../results/porto/porto_1000_model_results.csv

Running model_id=958, family=logistic_regression
CV AUC mean: 0.6198118772303183
CV AUC std: 0.004009598934704161
Saved progress to ../results/porto/porto_1000_model_results.csv

Running model_id=959, family=logistic_regression
CV AUC mean: 0.6198119423050319
CV AUC std: 0.004009781460599489
Saved progress to ../results/porto/porto_1000_model_results.csv

Running model_id=960, family=logistic_regression
CV AUC mean: 0.6198387226387633
CV AUC std: 0.004021587078046848
Saved progress to ../results/porto/porto_1000_model_results.csv

Running model_id=961, family=logistic_regression
CV AUC mean: 0.619723044572073
CV AUC std: 0.003903619100957689
Saved progress to ../results/porto/porto_1000_model_results.csv

Running model_id=962, family=logistic_regression
CV AUC mean: 0.6196778123885563
CV AUC std: 0.0037762127657963557
Saved progress to ../results/porto/porto_

In [18]:
results_df = pd.read_csv(RESULTS_CSV)

print("Results shape:", results_df.shape)
display(results_df.tail())

print("\nModel family counts:")
display(results_df["model_family"].value_counts())

print("\nSubmission status:")
display(results_df["submitted_to_kaggle"].value_counts(dropna=False))

Results shape: (1000, 18)


,competition,model_id,model_family,seed,params_json,started_at,cv_auc_mean,cv_auc_std,cv_auc_min,cv_auc_max,submission_path,submitted_to_kaggle,kaggle_status,public_auc,private_auc,runtime_seconds,error,finished_at
995,porto,995,logistic_regression,254394,"{""C"": 0.03, ""penalty"": ""l2"", ""solver"": ""liblin...",2026-05-13T17:08:13.503165,0.619822,0.004008,0.612650,0.624779,/Users/kunalgurung/Desktop/UTS SEM4/36126/proj...,False,NaN,NaN,NaN,11.698397,NaN,2026-05-13T17:08:25.201552
996,porto,996,logistic_regression,186747,"{""C"": 0.3, ""penalty"": ""l1"", ""solver"": ""libline...",2026-05-13T17:08:25.215819,0.619839,0.004021,0.612639,0.624803,/Users/kunalgurung/Desktop/UTS SEM4/36126/proj...,False,NaN,NaN,NaN,123.355560,NaN,2026-05-13T17:10:28.571367
997,porto,997,logistic_regression,236861,"{""C"": 0.01, ""penalty"": ""l2"", ""solver"": ""lbfgs""...",2026-05-13T17:10:28.587527,0.619742,0.003903,0.612690,0.624499,/Users/kunalgurung/Desktop/UTS SEM4/36126/proj...,False,NaN,NaN,NaN,7.455822,NaN,2026-05-13T17:10:36.043340
998,porto,998,logistic_regression,308127,"{""C"": 3.0, ""penalty"": ""l2"", ""solver"": ""libline...",2026-05-13T17:10:36.057255,0.619811,0.004009,0.612638,0.624774,/Users/kunalgurung/Desktop/UTS SEM4/36126/proj...,False,NaN,NaN,NaN,11.054031,NaN,2026-05-13T17:10:47.111275
999,porto,999,logistic_regression,800641,"{""C"": 3.0, ""penalty"": ""l1"", ""solver"": ""libline...",2026-05-13T17:10:47.125450,0.619814,0.004011,0.612639,0.624777,/Users/kunalgurung/Desktop/UTS SEM4/36126/proj...,False,NaN,NaN,NaN,16.241873,NaN,2026-05-13T17:11:03.367315



Model family counts:


model_family
lightgbm               350
xgboost                350
catboost               150
logistic_regression     75
extra_trees             42
random_forest           33
Name: count, dtype: int64


Submission status:


submitted_to_kaggle
False    1000
Name: count, dtype: int64

## 11. Validation Reliability Metrics

Once public and private leaderboard scores are available, calculate:

- CV bias: `cv_auc_mean - private_auc`
- Public leaderboard bias: `public_auc - private_auc`
- Absolute CV-private gap
- Absolute public-private gap

These metrics are the empirical foundation for the Bayesian melding model.

In [55]:
results_df = pd.read_csv(RESULTS_CSV)

results_df["cv_private_gap"] = results_df["cv_auc_mean"] - results_df["private_auc"]
results_df["abs_cv_private_gap"] = results_df["cv_private_gap"].abs()

results_df["public_private_gap"] = results_df["public_auc"] - results_df["private_auc"]
results_df["abs_public_private_gap"] = results_df["public_private_gap"].abs()

results_df.to_csv(RESULTS_CSV, index=False)

scored_df = results_df.dropna(subset=["public_auc", "private_auc"]).copy()

print("Total results:", len(results_df))
print("Rows with Kaggle public/private scores:", len(scored_df))

display(scored_df.head())

Total results: 700
Rows with Kaggle public/private scores: 1


,competition,model_id,model_family,seed,params_json,started_at,cv_auc_mean,cv_auc_std,cv_auc_min,cv_auc_max,...,kaggle_status,public_auc,private_auc,runtime_seconds,error,finished_at,cv_private_gap,abs_cv_private_gap,public_private_gap,abs_public_private_gap
5,santander,5,lightgbm,759899,"{""n_estimators"": 300, ""learning_rate"": 0.02, ""...",2026-05-09T21:44:22.964176,0.842029,0.003576,0.83614,0.846747,...,COMPLETE,0.84053,0.83816,124.96117,NaN,2026-05-09T21:46:27.925343,0.003869,0.003869,0.00237,0.00237


In [46]:
if len(scored_df) > 0:
    summary_by_family = scored_df.groupby("model_family").agg(
        n_models=("model_id", "count"),
        mean_cv_auc=("cv_auc_mean", "mean"),
        mean_public_auc=("public_auc", "mean"),
        mean_private_auc=("private_auc", "mean"),
        mean_cv_private_gap=("cv_private_gap", "mean"),
        mean_abs_cv_private_gap=("abs_cv_private_gap", "mean"),
        mean_public_private_gap=("public_private_gap", "mean"),
        mean_abs_public_private_gap=("abs_public_private_gap", "mean"),
        mean_cv_std=("cv_auc_std", "mean")
    ).reset_index()

    display(summary_by_family)

    summary_path = RESULTS_DIR / "porto_summary_by_family.csv"
    summary_by_family.to_csv(summary_path, index=False)
    print("Saved:", summary_path)
else:
    print("No scored Kaggle submissions yet.")

,model_family,n_models,mean_cv_auc,mean_public_auc,mean_private_auc,mean_cv_private_gap,mean_abs_cv_private_gap,mean_public_private_gap,mean_abs_public_private_gap,mean_cv_std
0,lightgbm,1,0.842029,0.84053,0.83816,0.003869,0.003869,0.00237,0.00237,0.003576


Saved: ../results/santander/santander_summary_by_family.csv


In [47]:
if len(scored_df) >= 3:
    corr_cols = ["cv_auc_mean", "public_auc", "private_auc"]
    corr_matrix = scored_df[corr_cols].corr()

    print("Correlation matrix:")
    display(corr_matrix)

    corr_path = RESULTS_DIR / "porto_score_correlations.csv"
    corr_matrix.to_csv(corr_path)
    print("Saved:", corr_path)
else:
    print("Need at least 3 scored submissions for correlation analysis.")

Need at least 3 scored submissions for correlation analysis.


In [ ]:
if len(scored_df) > 0:
    plt.figure(figsize=(8, 6))
    plt.scatter(scored_df["cv_auc_mean"], scored_df["private_auc"])
    plt.xlabel("5-Fold CV AUC")
    plt.ylabel("Private Leaderboard AUC")
    plt.title("Porto Seguro: CV AUC vs Private Leaderboard AUC")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(8, 6))
    plt.scatter(scored_df["public_auc"], scored_df["private_auc"])
    plt.xlabel("Public Leaderboard AUC")
    plt.ylabel("Private Leaderboard AUC")
    plt.title("Porto Seguro: Public vs Private Leaderboard AUC")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(8, 6))
    plt.hist(scored_df["cv_private_gap"], bins=20)
    plt.xlabel("CV AUC - Private AUC")
    plt.ylabel("Count")
    plt.title("Porto Seguro: CV Bias Distribution")
    plt.grid(True)
    plt.tight_layout()
    plt.show()
else:
    print("No scored rows available for plotting.")

In [15]:
results_df = pd.read_csv(RESULTS_CSV)

bayesian_cols = [
    "competition",
    "model_id",
    "model_family",
    "seed",
    "params_json",
    "cv_auc_mean",
    "cv_auc_std",
    "cv_auc_min",
    "cv_auc_max",
    "public_auc",
    "private_auc",
    "cv_private_gap",
    "abs_cv_private_gap",
    "public_private_gap",
    "abs_public_private_gap",
    "runtime_seconds"
]

available_cols = [c for c in bayesian_cols if c in results_df.columns]

bayesian_df = results_df[available_cols].copy()

bayesian_output_path = RESULTS_DIR / "porto_bayesian_melding_input.csv"
bayesian_df.to_csv(bayesian_output_path, index=False)

print("Saved Bayesian melding input dataset to:")
print(bayesian_output_path)

display(bayesian_df.head())

Saved Bayesian melding input dataset to:
../results/porto/porto_bayesian_melding_input.csv


,competition,model_id,model_family,seed,params_json,cv_auc_mean,cv_auc_std,cv_auc_min,cv_auc_max,public_auc,private_auc,runtime_seconds
0,porto,0,lightgbm,89251,"{""n_estimators"": 900, ""learning_rate"": 0.05, ""...",0.632019,0.002283,0.628338,0.634741,NaN,NaN,52.028652
1,porto,1,lightgbm,526479,"{""n_estimators"": 1100, ""learning_rate"": 0.05, ...",0.620765,0.004653,0.615249,0.628185,NaN,NaN,127.187861
2,porto,2,lightgbm,500352,"{""n_estimators"": 500, ""learning_rate"": 0.01, ""...",0.638708,0.003043,0.632972,0.641523,NaN,NaN,56.436534
3,porto,3,lightgbm,450460,"{""n_estimators"": 500, ""learning_rate"": 0.01, ""...",0.639645,0.002998,0.633748,0.642105,NaN,NaN,55.153564
4,porto,4,lightgbm,165229,"{""n_estimators"": 900, ""learning_rate"": 0.05, ""...",0.633354,0.002072,0.630247,0.636679,NaN,NaN,56.306771


In [20]:
results_df = pd.read_csv(RESULTS_CSV)

print("Shape:", results_df.shape)

display(results_df["model_family"].value_counts())

summary = results_df.groupby("model_family").agg(
    n_models=("model_id", "count"),
    mean_cv=("cv_auc_mean", "mean"),
    std_cv=("cv_auc_mean", "std"),
    min_cv=("cv_auc_mean", "min"),
    max_cv=("cv_auc_mean", "max"),
    mean_fold_std=("cv_auc_std", "mean"),
    mean_runtime=("runtime_seconds", "mean")
).reset_index()

display(summary.sort_values("mean_cv", ascending=False))

display(
    results_df.sort_values("cv_auc_mean", ascending=False)
    [["model_id", "model_family", "cv_auc_mean", "cv_auc_std", "runtime_seconds"]]
    .head(20)
)

Shape: (1000, 18)


model_family
lightgbm               350
xgboost                350
catboost               150
logistic_regression     75
extra_trees             42
random_forest           33
Name: count, dtype: int64

,model_family,n_models,mean_cv,std_cv,min_cv,max_cv,mean_fold_std,mean_runtime
2,lightgbm,350,0.636310,0.005954,0.604850,0.641269,0.003018,67.304881
5,xgboost,350,0.636236,0.007084,0.600862,0.642036,0.003072,59.705482
0,catboost,150,0.636034,0.003869,0.622503,0.640776,0.002976,116.093830
3,logistic_regression,75,0.619467,0.001297,0.615128,0.620361,0.003947,36.919149
1,extra_trees,42,0.618758,0.006111,0.598219,0.626398,0.003024,351.433838
4,random_forest,33,0.613508,0.006876,0.598015,0.623735,0.002839,508.611378


,model_id,model_family,cv_auc_mean,cv_auc_std,runtime_seconds
400,400,xgboost,0.642036,0.003095,93.896449
595,595,xgboost,0.641760,0.003089,102.013901
458,458,xgboost,0.641675,0.003084,102.890516
360,360,xgboost,0.641668,0.003438,80.685670
560,560,xgboost,0.641595,0.003391,89.448694
451,451,xgboost,0.641522,0.003044,78.269760
362,362,xgboost,0.641505,0.003039,44.558069
391,391,xgboost,0.641499,0.003310,90.502346
675,675,xgboost,0.641491,0.003098,83.584367
538,538,xgboost,0.641411,0.003512,80.168137


## 12. Representative Leaderboard Subset Selection

This section selects a representative subset of 100 Porto models for leaderboard evaluation.

The subset is selected across model families and cross-validation performance ranges rather than taking only the best-performing models. This reduces cherry-picking risk and supports fairer validation reliability analysis.

Random Forest and ExtraTrees are grouped into a combined `bagging_trees` category for subset allocation.

In [27]:
results_df = pd.read_csv(RESULTS_CSV)

# Keep only rows with generated submission files.
available_df = results_df[
    results_df["submission_path"].notna()
].copy()

# Treat Random Forest and ExtraTrees as one combined bagging family.
available_df["submission_family"] = available_df["model_family"].replace({
    "random_forest": "bagging_trees",
    "extra_trees": "bagging_trees"
})

family_quota = {
    "lightgbm": 25,
    "xgboost": 25,
    "catboost": 20,
    "bagging_trees": 15,
    "logistic_regression": 15
}

def select_top_middle_bottom(group_df, n_select):
    group_df = group_df.sort_values("cv_auc_mean").copy()

    n_bottom = n_select // 3
    n_top = n_select // 3
    n_middle = n_select - n_bottom - n_top

    bottom = group_df.head(n_bottom)
    top = group_df.tail(n_top)

    median_cv = group_df["cv_auc_mean"].median()
    middle = (
        group_df
        .assign(distance_to_median=(group_df["cv_auc_mean"] - median_cv).abs())
        .sort_values("distance_to_median")
        .head(n_middle)
    )

    selected = pd.concat([top, middle, bottom], ignore_index=True)
    selected = selected.drop_duplicates(subset=["model_id"])

    return selected

selected_parts = []

for family, quota in family_quota.items():
    family_df = available_df[available_df["submission_family"] == family].copy()

    selected_family = select_top_middle_bottom(
        group_df=family_df,
        n_select=quota
    )

    selected_parts.append(selected_family)

manual_submission_plan = (
    pd.concat(selected_parts, ignore_index=True)
    .drop_duplicates(subset=["model_id"])
    .sort_values(["submission_family", "cv_auc_mean"], ascending=[True, False])
)

manual_submission_plan_path = RESULTS_DIR / "manual_submission_plan_100.csv"
manual_submission_plan.to_csv(manual_submission_plan_path, index=False)

print("Manual submission plan size:", len(manual_submission_plan))
print("Saved to:", manual_submission_plan_path)

display(
    manual_submission_plan[
        [
            "model_id",
            "model_family",
            "submission_family",
            "cv_auc_mean",
            "cv_auc_std",
            "submission_path"
        ]
    ]
)

Manual submission plan size: 100
Saved to: ../results/porto/manual_submission_plan_100.csv


,model_id,model_family,submission_family,cv_auc_mean,cv_auc_std,submission_path
74,920,extra_trees,bagging_trees,0.626398,0.003539,/Users/kunalgurung/Desktop/UTS SEM4/36126/proj...
73,897,extra_trees,bagging_trees,0.625986,0.003227,/Users/kunalgurung/Desktop/UTS SEM4/36126/proj...
72,893,extra_trees,bagging_trees,0.625826,0.002708,/Users/kunalgurung/Desktop/UTS SEM4/36126/proj...
71,917,extra_trees,bagging_trees,0.625174,0.003290,/Users/kunalgurung/Desktop/UTS SEM4/36126/proj...
70,904,extra_trees,bagging_trees,0.624650,0.003028,/Users/kunalgurung/Desktop/UTS SEM4/36126/proj...
...,...,...,...,...,...,...
46,598,xgboost,xgboost,0.609230,0.002396,/Users/kunalgurung/Desktop/UTS SEM4/36126/proj...
45,551,xgboost,xgboost,0.608306,0.004126,/Users/kunalgurung/Desktop/UTS SEM4/36126/proj...
44,482,xgboost,xgboost,0.603608,0.001907,/Users/kunalgurung/Desktop/UTS SEM4/36126/proj...
43,522,xgboost,xgboost,0.601667,0.003248,/Users/kunalgurung/Desktop/UTS SEM4/36126/proj...


## 13. Manual Leaderboard Score Integration

This section records the manually verified Kaggle public and private leaderboard scores for the selected Porto representative models.

The Porto Seguro competition reports leaderboard performance using Normalized Gini. For consistency with the other competitions, the manually recorded Gini scores are converted to AUC using:

`AUC = (Gini + 1) / 2`

The converted AUC scores are saved together with the original Gini scores in `porto_leaderboard_scores.csv`.

In [53]:
# MANUAL PORTO LEADERBOARD SCORES
# Kaggle leaderboard scores are Normalized Gini.
# Values below are stored first as public_gini and private_gini.

leaderboard_records = [
    # model_id, public_gini, private_gini
    [2, 0.27412, 0.27880],
    [13, 0.27887, 0.28305],
    [14, 0.27853, 0.28362],
    [44, 0.27293, 0.27700],
    [51, 0.22161, 0.22310],
    [69, 0.21277, 0.21569],
    [85, 0.27514, 0.27913],
    [139, 0.27604, 0.28126],
    [158, 0.27858, 0.28344],
    [173, 0.22633, 0.22772],
    [204, 0.27603, 0.28065],
    [211, 0.22446, 0.22528],
    [237, 0.27898, 0.28398],
    [251, 0.21387, 0.21427],
    [261, 0.27955, 0.28334],
    [262, 0.27727, 0.28090],
    [272, 0.27862, 0.28312],
    [276, 0.27459, 0.27969],
    [285, 0.27357, 0.27803],
    [290, 0.21918, 0.22197],
    [300, 0.22965, 0.22791],
    [304, 0.27500, 0.28048],
    [316, 0.27879, 0.28357],
    [338, 0.22976, 0.22444],
    [349, 0.27535, 0.27977],
    [360, 0.27999, 0.28295],
    [362, 0.27920, 0.28365],
    [391, 0.27941, 0.28425],
    [400, 0.27958, 0.28417],
    [407, 0.22592, 0.22696],
    [410, 0.27505, 0.27948],
    [448, 0.27819, 0.28250],
    [451, 0.27914, 0.28283],
    [458, 0.27980, 0.28370],
    [482, 0.21489, 0.21438],
    [493, 0.22554, 0.22207],
    [494, 0.22300, 0.22174],
    [501, 0.27532, 0.28059],
    [522, 0.20764, 0.20332],
    [524, 0.20149, 0.20163],
    [551, 0.22607, 0.22023],
    [556, 0.27565, 0.27987],
    [557, 0.27573, 0.27931],
    [560, 0.27949, 0.28401],
    [577, 0.27526, 0.28005],
    [579, 0.27579, 0.27981],
    [589, 0.27567, 0.27999],
    [595, 0.27936, 0.28355],
    [598, 0.22386, 0.22170],
    [686, 0.27496, 0.28005],
    [702, 0.27588, 0.28106],
    [709, 0.25259, 0.25541],
    [724, 0.25582, 0.26087],
    [728, 0.27153, 0.27610],
    [738, 0.27237, 0.27592],
    [746, 0.27582, 0.28211],
    [747, 0.27155, 0.27563],
    [749, 0.25050, 0.25289],
    [779, 0.27292, 0.27725],
    [780, 0.27635, 0.28224],
    [782, 0.27620, 0.28095],
    [793, 0.25122, 0.25284],
    [818, 0.27121, 0.27504],
    [822, 0.24265, 0.24482],
    [831, 0.27177, 0.27596],
    [832, 0.27594, 0.28051],
    [834, 0.27126, 0.27785],
    [844, 0.25310, 0.25523],
    [847, 0.27648, 0.28250],
    [849, 0.27249, 0.27498],
    [852, 0.20352, 0.20451],
    [856, 0.19605, 0.20267],
    [879, 0.22555, 0.22994],
    [881, 0.22521, 0.23213],
    [885, 0.20156, 0.20212],
    [887, 0.19393, 0.19374],
    [892, 0.22785, 0.23170],
    [893, 0.24918, 0.25096],
    [897, 0.24845, 0.25078],
    [904, 0.24400, 0.25038],
    [913, 0.22684, 0.23230],
    [917, 0.24549, 0.24873],
    [920, 0.24876, 0.25116],
    [921, 0.18902, 0.19300],
    [924, 0.22846, 0.22997],
    [927, 0.23091, 0.23351],
    [929, 0.23807, 0.24308],
    [942, 0.23091, 0.23351],
    [943, 0.23091, 0.23351],
    [953, 0.23091, 0.23351],
    [956, 0.23740, 0.24279],
    [957, 0.23807, 0.24308],
    [958, 0.23740, 0.24279],
    [959, 0.23740, 0.24279],
    [963, 0.23091, 0.23351],
    [972, 0.23807, 0.24308],
    [984, 0.23740, 0.24279],
    [989, 0.23740, 0.24279],
    [991, 0.23807, 0.24308],
    [994, 0.23807, 0.24308],
]

leaderboard_df = pd.DataFrame(
    leaderboard_records,
    columns=["model_id", "public_gini", "private_gini"]
)

leaderboard_df = (
    leaderboard_df
    .drop_duplicates(subset=["model_id"], keep="first")
    .sort_values("model_id")
    .reset_index(drop=True)
)

# Convert Normalized Gini to AUC.
leaderboard_df["public_auc"] = (leaderboard_df["public_gini"] + 1) / 2
leaderboard_df["private_auc"] = (leaderboard_df["private_gini"] + 1) / 2

leaderboard_path = RESULTS_DIR / "porto_leaderboard_scores.csv"

leaderboard_df.to_csv(
    leaderboard_path,
    index=False
)

print("Porto leaderboard rows:", len(leaderboard_df))
print("Unique model IDs:", leaderboard_df["model_id"].nunique())
print("Saved leaderboard scores to:", leaderboard_path)

display(leaderboard_df.head())
display(leaderboard_df.tail())

Porto leaderboard rows: 100
Unique model IDs: 100
Saved leaderboard scores to: ../results/porto/porto_leaderboard_scores.csv


,model_id,public_gini,private_gini,public_auc,private_auc
0,2,0.27412,0.27880,0.637060,0.639400
1,13,0.27887,0.28305,0.639435,0.641525
2,14,0.27853,0.28362,0.639265,0.641810
3,44,0.27293,0.27700,0.636465,0.638500
4,51,0.22161,0.22310,0.610805,0.611550


,model_id,public_gini,private_gini,public_auc,private_auc
95,972,0.23807,0.24308,0.619035,0.621540
96,984,0.23740,0.24279,0.618700,0.621395
97,989,0.23740,0.24279,0.618700,0.621395
98,991,0.23807,0.24308,0.619035,0.621540
99,994,0.23807,0.24308,0.619035,0.621540


## 14. Final Porto Research Dataset Construction

This section merges the full cross-validation results with the manually verified leaderboard scores.

It then computes the final validation reliability metrics:

- `cv_private_gap`
- `abs_cv_private_gap`
- `public_private_gap`
- `abs_public_private_gap`

The output is saved as `porto_final_research_dataset.csv`.

In [54]:
# LOAD FULL CV RESULTS
results_df = pd.read_csv(RESULTS_CSV)

# REMOVE OLD/INCOMPLETE LEADERBOARD SCORE COLUMNS BEFORE MERGE
old_score_cols = [
    "public_auc",
    "private_auc",
    "public_gini",
    "private_gini",
    "cv_private_gap",
    "abs_cv_private_gap",
    "public_private_gap",
    "abs_public_private_gap"
]

results_df = results_df.drop(
    columns=[c for c in old_score_cols if c in results_df.columns],
    errors="ignore"
)

# MERGE CV RESULTS WITH CLEAN MANUAL LEADERBOARD SCORES
final_df = results_df.merge(
    leaderboard_df,
    on="model_id",
    how="inner"
)

# CREATE VALIDATION RELIABILITY METRICS
final_df["cv_private_gap"] = final_df["cv_auc_mean"] - final_df["private_auc"]
final_df["abs_cv_private_gap"] = final_df["cv_private_gap"].abs()

final_df["public_private_gap"] = final_df["public_auc"] - final_df["private_auc"]
final_df["abs_public_private_gap"] = final_df["public_private_gap"].abs()

final_df["competition"] = "porto"

# SAVE FINAL DATASET
final_dataset_path = RESULTS_DIR / "porto_final_research_dataset.csv"

final_df.to_csv(
    final_dataset_path,
    index=False
)

print("Final Porto research dataset rows:", len(final_df))
print("Saved final dataset to:", final_dataset_path)

display(
    final_df[
        [
            "competition",
            "model_id",
            "model_family",
            "cv_auc_mean",
            "cv_auc_std",
            "public_gini",
            "private_gini",
            "public_auc",
            "private_auc",
            "cv_private_gap",
            "public_private_gap"
        ]
    ].head()
)

display(
    final_df[
        [
            "competition",
            "model_id",
            "model_family",
            "cv_auc_mean",
            "cv_auc_std",
            "public_gini",
            "private_gini",
            "public_auc",
            "private_auc",
            "cv_private_gap",
            "public_private_gap"
        ]
    ].tail()
)

Final Porto research dataset rows: 100
Saved final dataset to: ../results/porto/porto_final_research_dataset.csv


,competition,model_id,model_family,cv_auc_mean,cv_auc_std,public_gini,private_gini,public_auc,private_auc,cv_private_gap,public_private_gap
0,porto,2,lightgbm,0.638708,0.003043,0.27412,0.27880,0.637060,0.639400,-0.000692,-0.002340
1,porto,13,lightgbm,0.641269,0.003101,0.27887,0.28305,0.639435,0.641525,-0.000256,-0.002090
2,porto,14,lightgbm,0.640848,0.002923,0.27853,0.28362,0.639265,0.641810,-0.000962,-0.002545
3,porto,44,lightgbm,0.638674,0.003594,0.27293,0.27700,0.636465,0.638500,0.000174,-0.002035
4,porto,51,lightgbm,0.609229,0.002346,0.22161,0.22310,0.610805,0.611550,-0.002321,-0.000745


,competition,model_id,model_family,cv_auc_mean,cv_auc_std,public_gini,private_gini,public_auc,private_auc,cv_private_gap,public_private_gap
95,porto,972,logistic_regression,0.620361,0.004108,0.23807,0.24308,0.619035,0.621540,-0.001179,-0.002505
96,porto,984,logistic_regression,0.619812,0.004010,0.23740,0.24279,0.618700,0.621395,-0.001583,-0.002695
97,porto,989,logistic_regression,0.619812,0.004010,0.23740,0.24279,0.618700,0.621395,-0.001583,-0.002695
98,porto,991,logistic_regression,0.620361,0.004108,0.23807,0.24308,0.619035,0.621540,-0.001179,-0.002505
99,porto,994,logistic_regression,0.620361,0.004108,0.23807,0.24308,0.619035,0.621540,-0.001179,-0.002505


## 15. Final Dataset Validation Checks

This section checks the final Porto research dataset for row counts, missing leaderboard scores, model-family coverage, and Gini-to-AUC conversion consistency.

In [55]:
print("Rows:", len(final_df))
print("Unique model IDs:", final_df["model_id"].nunique())
print("Missing public AUC:", final_df["public_auc"].isna().sum())
print("Missing private AUC:", final_df["private_auc"].isna().sum())
print("Missing public Gini:", final_df["public_gini"].isna().sum())
print("Missing private Gini:", final_df["private_gini"].isna().sum())

display(final_df["model_family"].value_counts())

display(
    final_df[
        [
            "model_id",
            "model_family",
            "cv_auc_mean",
            "public_gini",
            "private_gini",
            "public_auc",
            "private_auc",
            "cv_private_gap"
        ]
    ].sort_values("model_id").head(10)
)

Rows: 100
Unique model IDs: 100
Missing public AUC: 0
Missing private AUC: 0
Missing public Gini: 0
Missing private Gini: 0


model_family
lightgbm               25
xgboost                25
catboost               20
logistic_regression    15
extra_trees             9
random_forest           6
Name: count, dtype: int64

,model_id,model_family,cv_auc_mean,public_gini,private_gini,public_auc,private_auc,cv_private_gap
0,2,lightgbm,0.638708,0.27412,0.27880,0.637060,0.639400,-0.000692
1,13,lightgbm,0.641269,0.27887,0.28305,0.639435,0.641525,-0.000256
2,14,lightgbm,0.640848,0.27853,0.28362,0.639265,0.641810,-0.000962
3,44,lightgbm,0.638674,0.27293,0.27700,0.636465,0.638500,0.000174
4,51,lightgbm,0.609229,0.22161,0.22310,0.610805,0.611550,-0.002321
5,69,lightgbm,0.606622,0.21277,0.21569,0.606385,0.607845,-0.001223
6,85,lightgbm,0.638740,0.27514,0.27913,0.637570,0.639565,-0.000825
7,139,lightgbm,0.638742,0.27604,0.28126,0.638020,0.640630,-0.001888
8,158,lightgbm,0.640820,0.27858,0.28344,0.639290,0.641720,-0.000900
9,173,lightgbm,0.610377,0.22633,0.22772,0.613165,0.613860,-0.003483


## Summary

This notebook produces the Porto component of the final research dataset.

The final Porto dataset contains cross-validation scores, manually verified public/private leaderboard scores, original Normalized Gini scores, converted AUC scores, validation gap metrics, model family labels, and model configuration metadata.

This output is later combined with the Santander and Homesite experiment outputs in:

`04_final_analysis_bayesian_melding.ipynb`